# Simple PID loop double check
Craig Lage - 30Mar26

In [ ]:
import numpy as np
import pickle as pkl
import pandas as pd
import matplotlib.pyplot as plt

## Get the PID log.

In [ ]:
day_obs = 20260329

# Load the PID logs
# This was created with PID_Corrections_19Mar26.ipynb
filename = f"/home/c/cslage/u/MTAOS/times_square_reports/PID_data_{day_obs}.pkl"
with open(filename, 'rb') as f:
    pidDict = pkl.load(f)
    

In [ ]:
## Naming conventions

In [ ]:
all_labels = ['M2 dz', 'M2 dx', 'M2 dy', 'M2 rx', 'M2 ry',
     'Cam dz', 'Cam dx', 'Cam dy', 'Cam rx', 'Cam ry',
     'M1M3-1', 'M1M3-2', 'M1M3-3', 'M1M3-4', 'M1M3-5',
     'M1M3-6', 'M1M3-7', 'M1M3-8', 'M1M3-9', 'M1M3-10',
     'M1M3-11', 'M1M3-12', 'M1M3-13', 'M1M3-14', 'M1M3-15',
     'M1M3-16', 'M1M3-17', 'M1M3-18', 'M1M3-19', 'M1M3-20',
     'M2-1', 'M2-2', 'M2-3', 'M2-4', 'M2-5',
     'M2-6', 'M2-7', 'M2-8', 'M2-9', 'M2-10',
     'M2-11', 'M2-12', 'M2-13', 'M2-14', 'M2-15',
     'M2-16', 'M2-17', 'M2-18', 'M2-19', 'M2-20'
     ]

vmode_labels = ['Vmode1\nM2 tilts -rx-ry', 'Vmode2\nM2 tilts -rx+ry', 
                'Vmode3\nCam tilts -rx+ry', 'Vmode4\nCam tilts rx+ry',
                'Vmode5\nZ4-Focus', 'Vmode6\nZ5-Astig-Oblique',
                'Vmode7\nZ6-Astig-Vert', 'Vmode8\nZ7-Coma-Vert',
                'Vmode9\nZ8-Coma-Horiz', 'Vmode10\nZ9-Trefoil-Vert',
                'Vmode11\nZ10-Trefoil-Oblique', 'Vmode12\nZ11-Spherical']


## Next I try to calculate the correction.  These now agree if I use the logged
## "Error" term as dof_visit.

In [ ]:
seqNum = 273
expId = int(1E5 * day_obs + seqNum)
dof_visit = pidDict[expId]['Error']  # From the "Error before any gains..." in the log message.
print(f"dof_visit seqNum{seqNum}")
print(dof_visit)
print()
correction = pidDict[expId]['Correction'] # From the "Correction" in the log message.
print(f"Correction {expId}")
print(correction)
print()
clip_i = pidDict[expId]['Clipped_I'] # From the "Clipped_I" in the log message.
calculated_correction = dof_visit * 0.18 + clip_i * 0.022
print(f"Calculated correction {expId}")
print(calculated_correction)
print("These are identical within a small error")

## Verifying agreement for the whole night.

In [ ]:
corrections = []
calculated_corrections = []
for expId in pidDict.keys():
    thisDict = pidDict[expId]
    correction = pidDict[expId]['Correction'] # The Correction term in the log
    corrections.append(correction)
    dof_visit = pidDict[expId]['Error'] # The error term in the log
    clip_i = pidDict[expId]['Clipped_I'] # The Clipped_I term in the log
    Kp = pidDict[expId]['Kp']
    Ki = pidDict[expId]['Ki']
    calculated_correction = dof_visit * Kp + clip_i * Ki
    calculated_corrections.append(calculated_correction)
    
corrections = np.array(corrections)
calculated_corrections = np.array(calculated_corrections)

indices = list(range(0, 17)) + list(range(30,35)) # This just collapses the 50 DOF down to the 22 we are using
fig, axes = plt.subplots(5, 5, figsize=(15,15))
plt.subplots_adjust(hspace=0.5, wspace = 0.6)
plt.suptitle(f"Logged corrections vs calculated corrections {day_obs}", fontsize=18, y=0.95)
for i in range(5):
    for j in range(5):
        index = i + 5 * j
        if index > 21:
            axes[i][j].set_axis_off()
            continue
        label = all_labels[indices[index]]
        axes[i][j].scatter(corrections[:,index], calculated_corrections[:,index])
        axes[i][j].set_title(label)
        axes[i][j].set_xlabel("Logged correction")
        axes[i][j].set_ylabel("Calculated correction")
        
fig.savefig(f"/home/c/cslage/u/MTAOS/times_square_reports/PID_Corrections_Check_{day_obs}.png", 
            bbox_inches='tight', pad_inches=0.1)